In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
print(os.listdir("/kaggle/input"))

In [ ]:
print(os.listdir("/kaggle/input/competitions"))

In [ ]:
comp_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
print(os.listdir(comp_dir))

# 03 — DICOM Exploration

**Purpose:** Answer the two priority questions that `02_data_exploration.ipynb` could not resolve from tabular data alone:

1. Does DICOM metadata provide a patient/site/scanner grouping key usable for validation?
2. Does one `StudyInstanceUID` represent one knee, or can a study contain both?

Everything else in this notebook (imaging hierarchy, dimensions, spacing, transfer syntax, decode reliability) supports those two questions or lays groundwork for `03b`/baseline work — it is not the primary goal.

**Out of scope:** preprocessing pipeline, dataset class, model training, report-to-label extraction, `src/` abstractions. This is still the understanding phase.

**Sensitive data handling:** if patient-identifying fields exist (PatientID, PatientName, PatientBirthDate, AccessionNumber), this notebook analyzes their *presence and uniqueness characteristics only* — actual values are never printed, plotted, or saved to output files.

## 1. Environment and Data Paths

In [ ]:
pip install pydicom

In [ ]:
import os
import glob
import pydicom
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

CSV_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
DICOM_CANDIDATE_ROOTS = [
    os.path.join(CSV_ROOT, "train_series"),
    "./data/train_series",
    "/kaggle/input/rsna-knee-abnormality-detection/train_series",
]

DICOM_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection/train_series"
for d in DICOM_CANDIDATE_ROOTS:
    if os.path.isdir(d) and len(os.listdir(d)) > 0:
        DICOM_ROOT = d
        break

if DICOM_ROOT is None:
    raise FileNotFoundError(
        "No DICOM data found in any candidate directory: "
        f"{DICOM_CANDIDATE_ROOTS}. This notebook cannot proceed without actual "
        "DICOM files. Download/mount the train_series/ folder before continuing "
        "-- do not proceed with placeholder or synthetic data."
    )

print("Resolved DICOM_ROOT:", os.path.abspath(DICOM_ROOT))

train = pd.read_csv(os.path.join(CSV_ROOT, "train.csv"))
train_series = pd.read_csv(os.path.join(CSV_ROOT, "train_series.csv"))
print(f"Loaded metadata: {len(train)} studies, {len(train_series)} series (from 02_data_exploration.ipynb source files)")

## 2. DICOM Dataset Inventory

In [ ]:
study_dirs = [d for d in os.listdir(DICOM_ROOT) if os.path.isdir(os.path.join(DICOM_ROOT, d))]
print("Study directories found on disk:", len(study_dirs))

series_counts = {}
file_counts = {}
for study in study_dirs:
    study_path = os.path.join(DICOM_ROOT, study)
    series_dirs = [d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d))]
    series_counts[study] = len(series_dirs)
    total_files = 0
    for series in series_dirs:
        total_files += len(glob.glob(os.path.join(study_path, series, "*.dcm")))
    file_counts[study] = total_files

print("Series per study on disk -- describe:")
print(pd.Series(series_counts).describe())
print("\nDICOM files per study on disk -- describe:")
print(pd.Series(file_counts).describe())

# Cross-reference disk contents against CSV metadata
csv_study_ids = set(train["StudyInstanceUID"])
disk_study_ids = set(study_dirs)

missing_on_disk = csv_study_ids - disk_study_ids
extra_on_disk = disk_study_ids - csv_study_ids

print(f"\nStudies in train.csv but NOT found on disk: {len(missing_on_disk)}")
print(f"Study directories on disk but NOT in train.csv: {len(extra_on_disk)}")

if missing_on_disk:
    print("NOTE: not every study may be downloaded locally -- this can be expected "
          "if only a partial dataset was fetched. Confirm before treating as a data-quality issue.")

## 3. Representative Study Selection

Sample deliberately across series-count, plane combinations, and study size rather than picking convenient examples.

In [ ]:
import random
random.seed(42)  # fixed seed for reproducibility of the sample itself

available_studies = [s for s in study_dirs if s in csv_study_ids]

# Bucket by series count (from train_series.csv, ground truth) to get spread
series_per_study_csv = train_series.groupby("StudyInstanceUID").size()
available_series_counts = series_per_study_csv.loc[series_per_study_csv.index.isin(available_studies)]

low = available_series_counts[available_series_counts <= available_series_counts.quantile(0.25)].index.tolist()
mid = available_series_counts[
    (available_series_counts > available_series_counts.quantile(0.25)) &
    (available_series_counts < available_series_counts.quantile(0.75))
].index.tolist()
high = available_series_counts[available_series_counts >= available_series_counts.quantile(0.75)].index.tolist()

SAMPLE_SIZE_PER_BUCKET = 3  # adjust based on how many studies are actually downloaded
sample_studies = (
    random.sample(low, min(SAMPLE_SIZE_PER_BUCKET, len(low))) +
    random.sample(mid, min(SAMPLE_SIZE_PER_BUCKET, len(mid))) +
    random.sample(high, min(SAMPLE_SIZE_PER_BUCKET, len(high)))
)

# Include labeled studies specifically, if any are downloaded, since they're the most valuable to inspect
labeled_ids = set(train.dropna(subset=[c for c in train.columns if c not in ("StudyInstanceUID", "Report")])["StudyInstanceUID"])
labeled_available = [s for s in available_studies if s in labeled_ids]
if labeled_available:
    sample_studies += random.sample(labeled_available, min(3, len(labeled_available)))

sample_studies = list(dict.fromkeys(sample_studies))  # de-dupe, preserve order
print(f"Selected {len(sample_studies)} studies for detailed inspection:")
for s in sample_studies:
    print(" ", s, "-- series (csv):", series_per_study_csv.get(s, "unknown"))

## 4. DICOM Header Inventory

Read headers only (`stop_before_pixels=True`) for a broad tag-presence pass; pixel data is loaded separately and selectively in later sections.

In [ ]:
IDENTITY_TAGS = ["PatientID", "PatientName", "PatientBirthDate", "PatientSex"]
STUDY_TAGS = ["StudyInstanceUID", "StudyDate", "StudyDescription", "AccessionNumber"]
SERIES_TAGS = ["SeriesInstanceUID", "SeriesNumber", "SeriesDescription", "ProtocolName"]
EQUIPMENT_TAGS = ["InstitutionName", "Manufacturer", "ManufacturerModelName", "StationName", "DeviceSerialNumber"]
LATERALITY_TAGS = ["ImageLaterality", "BodyPartExamined", "Laterality"]
GEOMETRY_TAGS = ["Rows", "Columns", "PixelSpacing", "SliceThickness", "SpacingBetweenSlices",
                 "ImageOrientationPatient", "ImagePositionPatient", "InstanceNumber"]

ALL_TAG_GROUPS = {
    "identity": IDENTITY_TAGS, "study": STUDY_TAGS, "series": SERIES_TAGS,
    "equipment": EQUIPMENT_TAGS, "laterality": LATERALITY_TAGS, "geometry": GEOMETRY_TAGS,
}

def inspect_header(path):
    """Read one DICOM header (no pixel data). Local helper -- promote to src/
    only once the tag set and interface are stable across this + future notebooks."""
    ds = pydicom.dcmread(path, stop_before_pixels=True)
    row = {"__path": path}
    for group, tags in ALL_TAG_GROUPS.items():
        for tag in tags:
            row[tag] = getattr(ds, tag, None)
    row["TransferSyntaxUID"] = str(ds.file_meta.TransferSyntaxUID) if hasattr(ds, "file_meta") else None
    return row

all_dcm_paths = []
for study in sample_studies:
    all_dcm_paths += glob.glob(os.path.join(DICOM_ROOT, study, "**", "*.dcm"), recursive=True)

print(f"Reading headers for {len(all_dcm_paths)} files across {len(sample_studies)} sampled studies...")
header_rows = [inspect_header(p) for p in all_dcm_paths]
header_df = pd.DataFrame(header_rows)

print("\nTag presence (non-null count / total):")
for col in [c for c in header_df.columns if c != "__path"]:
    present = header_df[col].notna().sum()
    print(f"  {col:28s}: {present}/{len(header_df)}")

## 5. Patient / Site / Validation Investigation

No raw identifiers are printed below -- presence and uniqueness only.

In [ ]:
def summarize_grouping_field(df, field, label):
    if field not in df.columns or df[field].notna().sum() == 0:
        print(f"{label} ({field}): NOT present / not populated in sampled headers.")
        return None
    n_unique = df[field].nunique()
    print(f"{label} ({field}): present. Unique values in sample: {n_unique} across {df[field].notna().sum()} populated rows.")
    return field

patient_field = summarize_grouping_field(header_df, "PatientID", "Patient identifier")
institution_field = summarize_grouping_field(header_df, "InstitutionName", "Institution")
manufacturer_field = summarize_grouping_field(header_df, "Manufacturer", "Scanner manufacturer")
model_field = summarize_grouping_field(header_df, "ManufacturerModelName", "Scanner model")
station_field = summarize_grouping_field(header_df, "StationName", "Station")
device_field = summarize_grouping_field(header_df, "DeviceSerialNumber", "Device serial")

if patient_field:
    # Attach StudyInstanceUID for the per-patient study-count analysis in Section 6,
    # without ever displaying PatientID values themselves.
    header_df["__study_from_path"] = header_df["__path"].apply(lambda p: p.split(os.sep)[-3] if os.sep in p else None)

## 6. Study ↔ Patient Relationship

Only runs meaningfully if Section 5 found a populated `PatientID` field.

In [ ]:
if patient_field:
    study_to_patient = header_df.dropna(subset=["PatientID"]).drop_duplicates("__study_from_path")[["__study_from_path", "PatientID"]]
    studies_per_patient = study_to_patient.groupby("PatientID").size()
    print("Studies per patient (sampled studies only -- not the full dataset):")
    print(studies_per_patient.describe())
    multi_study_patients = (studies_per_patient > 1).sum()
    print(f"\nPatients with more than one study in this sample: {multi_study_patients} / {studies_per_patient.shape[0]}")
    if multi_study_patients > 0:
        print("IMPLICATION: patient-level leakage is possible if validation splits by study "
              "rather than by patient. A grouped split (GroupKFold on PatientID) would be needed.")
    else:
        print("No repeated patients observed in this sample -- inconclusive with a small sample; "
              "re-run this section across a larger fraction of studies before concluding leakage risk is low.")
else:
    print("Skipped: no patient identifier field was found in Section 5.")

## 7. Laterality Investigation

Evidence from DICOM metadata tags. Visual/series-consistency evidence follows in Section 12.

In [ ]:
laterality_summary = header_df.groupby("__study_from_path" if "__study_from_path" in header_df.columns else "__path").agg({
    "ImageLaterality": lambda x: set(x.dropna()),
    "BodyPartExamined": lambda x: set(x.dropna()),
    "Laterality": lambda x: set(x.dropna()),
    "SeriesDescription": lambda x: set(x.dropna()),
    "StudyDescription": lambda x: set(x.dropna()),
})

print("Laterality-related tag values per sampled study:")
print(laterality_summary)

any_laterality_tag_populated = any(
    header_df[c].notna().any() for c in ["ImageLaterality", "Laterality"] if c in header_df.columns
)
print(f"\nAny explicit laterality tag populated across the sample: {any_laterality_tag_populated}")

if any_laterality_tag_populated:
    multi_side_studies = laterality_summary[
        laterality_summary["ImageLaterality"].apply(lambda s: len(s) > 1) |
        laterality_summary["Laterality"].apply(lambda s: len(s) > 1)
    ]
    print(f"Studies with more than one laterality value across their series: {len(multi_side_studies)}")
else:
    print("No explicit laterality tag found -- rely on StudyDescription/SeriesDescription text "
          "and visual inspection (Section 12) instead. Do not conclude 'one knee per study' from "
          "tag absence alone.")

## 8. DICOM Study / Series / Slice Structure

In [ ]:
def series_slice_info(study, series):
    paths = sorted(glob.glob(os.path.join(DICOM_ROOT, study, series, "*.dcm")))
    instance_numbers = []
    sop_uids = set()
    duplicate_sops = 0
    for p in paths:
        ds = pydicom.dcmread(p, stop_before_pixels=True)
        instance_numbers.append(getattr(ds, "InstanceNumber", None))
        sop = getattr(ds, "SOPInstanceUID", None)
        if sop in sop_uids:
            duplicate_sops += 1
        sop_uids.add(sop)
    return {
        "n_files": len(paths),
        "n_unique_sop": len(sop_uids),
        "duplicate_sop_count": duplicate_sops,
        "instance_numbers_sorted": sorted([n for n in instance_numbers if n is not None]) == \
                                    [n for n in instance_numbers if n is not None],
    }

slice_structure_rows = []
for study in sample_studies:
    study_path = os.path.join(DICOM_ROOT, study)
    if not os.path.isdir(study_path):
        continue
    for series in os.listdir(study_path):
        if os.path.isdir(os.path.join(study_path, series)):
            info = series_slice_info(study, series)
            info.update({"study": study, "series": series})
            slice_structure_rows.append(info)

slice_structure_df = pd.DataFrame(slice_structure_rows)
print(slice_structure_df)

if slice_structure_df["duplicate_sop_count"].sum() > 0:
    print("\nWARNING: duplicate SOPInstanceUIDs found -- investigate before assuming clean slice ordering.")

## 9. Image Dimensions and Pixel Spacing

In [ ]:
geometry_summary = header_df[["Rows", "Columns", "PixelSpacing", "SliceThickness",
                               "SpacingBetweenSlices", "TransferSyntaxUID"]].copy()

print("Rows/Columns distribution:")
print(geometry_summary[["Rows", "Columns"]].describe())

print("\nUnique (Rows, Columns) combinations observed:")
print(geometry_summary.groupby(["Rows", "Columns"]).size())

print("\nSliceThickness distribution:")
print(geometry_summary["SliceThickness"].describe())

print("\nPixelSpacing -- sample of raw values (arrays, not directly summarizable numerically):")
print(geometry_summary["PixelSpacing"].dropna().head(10))

## 10. Transfer Syntax and Decoding

In [ ]:
print("Transfer syntaxes observed in sample:")
print(header_df["TransferSyntaxUID"].value_counts(dropna=False))

decode_results = []
for p in all_dcm_paths:
    try:
        ds = pydicom.dcmread(p)
        arr = ds.pixel_array  # forces decode
        decode_results.append({"path": p, "status": "OK", "shape": arr.shape, "dtype": str(arr.dtype)})
    except Exception as e:
        decode_results.append({"path": p, "status": "FAIL", "error": f"{type(e).__name__}: {e}"})

decode_df = pd.DataFrame(decode_results)
print(f"\nDecode success: {(decode_df['status'] == 'OK').sum()} / {len(decode_df)}")
failures = decode_df[decode_df["status"] == "FAIL"]
if len(failures) > 0:
    print("\nDecode failures (do not silently skip -- investigate library requirements):")
    print(failures[["path", "error"]])

## 11. Pixel Data and Intensity Inspection

In [ ]:
intensity_rows = []
for p in all_dcm_paths[:20]:  # cap to keep this section fast; expand if needed
    try:
        ds = pydicom.dcmread(p)
        arr = ds.pixel_array
        intensity_rows.append({
            "path": os.path.basename(p),
            "shape": arr.shape,
            "dtype": str(arr.dtype),
            "min": arr.min(),
            "max": arr.max(),
            "p1": np.percentile(arr, 1),
            "p99": np.percentile(arr, 99),
        })
    except Exception:
        continue

intensity_df = pd.DataFrame(intensity_rows)
print(intensity_df)

## 12. Representative Visualization

In [ ]:
def show_study_planes(study):
    """Show one representative slice per plane for a study, using train_series.csv
    to identify which series corresponds to which plane."""
    study_series_meta = train_series[train_series["StudyInstanceUID"] == study]
    planes = study_series_meta["Anatomical_Plane"].dropna().unique()

    fig, axes = plt.subplots(1, len(planes), figsize=(5 * len(planes), 5))
    if len(planes) == 1:
        axes = [axes]

    for ax, plane in zip(axes, planes):
        series_id = study_series_meta[study_series_meta["Anatomical_Plane"] == plane]["SeriesInstanceUID"].iloc[0]
        series_path = os.path.join(DICOM_ROOT, study, series_id)
        dcm_paths = sorted(glob.glob(os.path.join(series_path, "*.dcm")))
        if not dcm_paths:
            continue
        mid_slice_path = dcm_paths[len(dcm_paths) // 2]
        ds = pydicom.dcmread(mid_slice_path)
        ax.imshow(ds.pixel_array, cmap="gray")
        ax.set_title(f"{plane}")
        ax.axis("off")

    fig.suptitle(f"Study {study[:20]}... -- {len(dcm_paths)} slices in shown series")
    plt.tight_layout()
    plt.show()

for study in sample_studies[:3]:  # keep this to a handful, per the "don't create dozens of plots" guidance
    show_study_planes(study)

## 13. DICOM Data Quality

In [ ]:
quality_summary = {
    "studies_missing_on_disk_vs_csv": len(missing_on_disk),
    "unexpected_studies_on_disk": len(extra_on_disk),
    "duplicate_sop_instances_in_sample": int(slice_structure_df["duplicate_sop_count"].sum()) if len(slice_structure_df) else None,
    "decode_failures_in_sample": int((decode_df["status"] == "FAIL").sum()) if len(decode_df) else None,
    "unsorted_instance_numbers_series_count": int((~slice_structure_df["instance_numbers_sorted"]).sum()) if len(slice_structure_df) else None,
}

print("DICOM data-quality summary (sampled studies only):")
for k, v in quality_summary.items():
    flag = "  <-- non-zero" if v not in (0, None) else ""
    print(f"  {k}: {v}{flag}")

## 14. Findings and Implications

### Confirmed Facts (full 4,407-study dataset)

- **Patient/validation:** `PatientID` present in 4,407/4,407 studies. 4,407 unique patients — exactly one study per patient, zero exceptions. Zero cross-series `PatientID` inconsistency. **Patient-repeat leakage is not a risk in this dataset.**
- **Laterality coverage:** ~50% (2,204/4,407 studies tagged). Of the tagged studies: 2,179 single-consistent-value, 25 with genuine L/R conflict across series, 1 explicitly tagged `B`.
- **Laterality missingness is manufacturer-determined, not random:** GE MEDICAL SYSTEMS, TOSHIBA, Philips Healthcare, and CANON_MEC (1,185 studies, 26.9% of the dataset) show 0% coverage; Siemens Healthineers, Siemens, GEHC, FUJIFILM, Hitachi show 99.9–100%; SIEMENS/Philips Medical Systems/Philips show partial (38–62%) coverage. No further sampling will recover this — it requires a different data source (e.g. report text) if needed.
- **`BodyPartExamined` is unreliable:** contains many non-knee values (BRAIN, HEART, LIVER, SPINE, etc.), almost certainly a stale-protocol-preset artifact common in clinical DICOM data rather than evidence of actual non-knee scans. Not to be used as a hard filter.
- **Slice ordering:** file-listing order does not reliably correspond to anatomical order in the sampled series. Future preprocessing must derive order from `InstanceNumber` where fully populated and unique, falling back to `ImagePositionPatient` projection otherwise — not from directory listing.

### 26 Laterality Exceptions — Visual Verification Results (final)

Of the 26 laterality-exception studies (25 L/R conflicts + 1 explicit `B` tag):

- **8 clearly bilateral** — direct visual confirmation (two distinct knee
  cross-sections visible in a single frame).
- **2 likely bilateral, moderate-high confidence** — no single dual-knee
  frame, but clean per-side series grouping + large geometric separation
  (~99-100mm between laterality-tagged series groups).
- **12 likely bilateral, lower confidence** — share a distinctive pattern:
  one minority-laterality series against 4-5 majority-laterality series,
  no single frame showing both knees. Could indicate a genuine but
  asymmetric bilateral acquisition (e.g. one comparison series from the
  opposite knee) or a single mistagged series. Not resolved with certainty.
- **3 metadata artifact / inconsistency** — small geometric separation
  and/or unrecognized laterality values, visually single-knee.
- **1 ambiguous** — inconclusive from available evidence.

This review was conducted via visual inspection of generated series images
cross-checked against DICOM `ImagePositionPatient` geometry. It was not
radiologist-grade and should be treated as a strong first pass, not a
clinical determination, for the "lower confidence" and "ambiguous" cases.

### Sample-Only Observations (from the original 9-study sample, Sections 2–13 — not yet verified at full-dataset scale)

- Image dimension variability (17 distinct Rows×Columns combos observed)
- SliceThickness range (0.6–4.0mm)
- Transfer syntax (only Explicit VR Little Endian observed in this sample — documentation indicates compressed syntaxes exist elsewhere in the dataset; not contradicted, just not sampled)
- Decode reliability (100% success in the 1,824-file sample)

### Validation Implications

Patient-grouped validation is **not required** to prevent patient-repeat leakage — verified conclusively at full-dataset scale. Other leakage vectors (near-duplicate imaging, site-clustering effects) remain untested and out of scope for this milestone.

### Target-Semantics Implications

`StudyInstanceUID → knee/laterality → 12 target labels`: for the ~97.7%
of studies with either no conflicting evidence or a single consistent
laterality value, treating each study as single-knee is a reasonable
default.

For the 26 flagged studies, visual verification (see above) resolved this
concretely:

- **10 studies (8 clearly bilateral + 2 moderate-high confidence)** should
  be treated as genuinely bilateral for modeling purposes. Target-label
  semantics for these need explicit handling — a single set of 12 labels
  per `StudyInstanceUID` may not cleanly map to "one knee" for this subset,
  and this should be revisited when designing the training pipeline.
- **12 studies (lower confidence)** are probably bilateral but not
  certain, given the distinctive imbalanced-series pattern observed. Safe
  default: treat as bilateral (same handling as the confirmed group) since
  that's the more conservative assumption, but flag these specifically if
  later evidence (e.g. report-text cross-check) can resolve them further.
- **4 studies (3 artifact + 1 ambiguous)** show no visual bilateral
  evidence despite the metadata conflict. Safe default: treat as
  single-knee, since the conflicting tag is more likely a data-quality
  issue than a real second knee.

In total: **~22 of 4,407 studies (0.50%) likely need bilateral-aware
handling**, ~4 studies' metadata conflicts are likely artifacts safely
ignorable, and the remaining ~99.4% of the dataset can be treated as
single-knee-per-study with reasonable confidence.

### Remaining DICOM Questions

- Whether compressed transfer syntaxes appear elsewhere in the full
  dataset (untested beyond the 9-study sample — only Explicit VR Little
  Endian observed so far).
- Whether the untagged ~50% laterality gap can be partially recovered from
  report text for the manufacturers with 0% DICOM laterality coverage.
- Whether the 12 "lower confidence" and 1 "ambiguous" bilateral
  classifications can be resolved further using report text or additional
  DICOM tags not yet inspected (e.g. `ProtocolName`, `SeriesDescription`
  patterns specific to the affected studies).
- Other leakage vectors beyond patient repeats (near-duplicate imaging,
  site-clustering effects) — not investigated in this milestone.

### Milestone 1C Status: Complete

All completion criteria are satisfied with executed evidence: full-dataset
header scan (4,407/4,407), patient/grouping resolution, laterality
coverage and manufacturer-dependency characterization, all 26 exception
studies visually classified, `BodyPartExamined` unreliability documented,
slice-ordering conclusion established, no sensitive identifiers exposed.
Next milestone: **Validation Strategy** — designing a trustworthy local
split for the 12-target multilabel problem with only 58 directly labeled
studies, informed by the findings above (no patient-repeat leakage risk;
~22 studies needing bilateral-aware target handling; manufacturer as a
potential distribution-shift factor to consider).

## Full-Dataset Header Validation

The sections above (2–13) established findings from a 9-study sample. Two questions matter too much for validation and target-semantics decisions to leave sample-based — this section extends them to all 4,407 training studies via header-only reads (`stop_before_pixels=True`, no pixel data loaded).

**Runtime note:** the patient/manufacturer pass reads one file per study (~4,407 reads). The laterality-consistency pass reads one file per *series* (~24,371 reads, since laterality must be checked across series within a study, not assumed from a single representative file) — this will take noticeably longer. Both remain header-only and far cheaper than any pixel-level scan.

### Full-Dataset Header Scan

In [ ]:
if "study_dirs" not in dir():
    study_dirs = [d for d in os.listdir(DICOM_ROOT) if os.path.isdir(os.path.join(DICOM_ROOT, d))]
    print(f"Rebuilt study_dirs: {len(study_dirs)} studies (state was reset)")

STUDY_LEVEL_TAGS = ["PatientID", "Laterality", "BodyPartExamined", "Manufacturer", "ManufacturerModelName"]

def get_representative_file(study_dir):
    series_dirs = sorted([d for d in os.listdir(study_dir) if os.path.isdir(os.path.join(study_dir, d))])
    if not series_dirs:
        return None
    files = sorted(glob.glob(os.path.join(study_dir, series_dirs[0], "*.dcm")))
    return files[0] if files else None

def read_tags(path, tags):
    ds = pydicom.dcmread(path, stop_before_pixels=True)
    return {t: getattr(ds, t, None) for t in tags}

study_rows = []
failed_reads = []
for study in study_dirs:  # full list of all 4,407 studies, from Section 2
    study_path = os.path.join(DICOM_ROOT, study)
    rep_file = get_representative_file(study_path)
    if rep_file is None:
        failed_reads.append(study)
        continue
    tags = read_tags(rep_file, STUDY_LEVEL_TAGS)
    tags["StudyInstanceUID"] = study
    study_rows.append(tags)

study_level_df = pd.DataFrame(study_rows)
print(f"Successfully read representative header for {len(study_level_df)} / {len(study_dirs)} studies")
if failed_reads:
    print(f"Studies with no readable representative file: {len(failed_reads)}")
    print(failed_reads[:20])

### Patient ↔ Study Analysis

Aggregate statistics only. No `PatientID` value is ever printed below.

In [ ]:
n_total = len(study_level_df)
n_with_patient = int(study_level_df["PatientID"].notna().sum())
print(f"PatientID present: {n_with_patient} / {n_total}")

patient_counts = study_level_df.dropna(subset=["PatientID"]).groupby("PatientID").size()
n_unique_patients = patient_counts.shape[0]
print(f"Unique patients: {n_unique_patients}")
print("\nStudies per patient -- describe:")
print(patient_counts.describe())

n_single = int((patient_counts == 1).sum())
n_multi = int((patient_counts > 1).sum())
n_two = int((patient_counts == 2).sum())
n_three_plus = int((patient_counts >= 3).sum())
max_studies = int(patient_counts.max()) if len(patient_counts) else 0
multi_patient_ids = patient_counts[patient_counts > 1].index
pct_multi_studies = (
    100 * study_level_df[study_level_df["PatientID"].isin(multi_patient_ids)].shape[0] / n_total
    if n_total else 0
)

print(f"\nPatients with exactly 1 study: {n_single}")
print(f"Patients with >1 study: {n_multi}")
print(f"  of which exactly 2 studies: {n_two}")
print(f"  of which 3+ studies: {n_three_plus}")
print(f"Max studies for a single patient: {max_studies}")
print(f"% of studies belonging to multi-study patients: {pct_multi_studies:.2f}%")

In [ ]:
# Series-level pass: needed to verify PatientID and Laterality consistency WITHIN a study,
# not just from one representative file. This is the more expensive pass (~24,371 reads).
SERIES_LEVEL_TAGS = ["PatientID", "Laterality", "BodyPartExamined"]

def get_first_file(series_path):
    files = sorted(glob.glob(os.path.join(series_path, "*.dcm")))
    return files[0] if files else None

series_rows = []
for study in study_dirs:
    study_path = os.path.join(DICOM_ROOT, study)
    if not os.path.isdir(study_path):
        continue
    for series in os.listdir(study_path):
        series_path = os.path.join(study_path, series)
        if not os.path.isdir(series_path):
            continue
        f = get_first_file(series_path)
        if f is None:
            continue
        tags = read_tags(f, SERIES_LEVEL_TAGS)
        tags["StudyInstanceUID"] = study
        tags["SeriesInstanceUID"] = series
        series_rows.append(tags)

series_level_df = pd.DataFrame(series_rows)
print(f"Read series-level header for {len(series_level_df)} series across {series_level_df['StudyInstanceUID'].nunique()} studies")

In [ ]:
# Consistency: does every study map to exactly one PatientID across all its series?
patient_per_study = series_level_df.groupby("StudyInstanceUID")["PatientID"].apply(lambda x: set(x.dropna()))
inconsistent_patient_studies = patient_per_study[patient_per_study.apply(len) > 1]

print(f"Studies with inconsistent PatientID across series: {len(inconsistent_patient_studies)}")
if len(inconsistent_patient_studies) > 0:
    print("WARNING: study -> PatientID is not 1:1 for these studies (StudyInstanceUID only, no PatientID shown):")
    print(list(inconsistent_patient_studies.index))

### Laterality ↔ Study Analysis

In [ ]:
laterality_per_study = series_level_df.groupby("StudyInstanceUID")["Laterality"].apply(lambda x: set(x.dropna()))
bodypart_per_study = series_level_df.groupby("StudyInstanceUID")["BodyPartExamined"].apply(lambda x: set(x.dropna()))

n_no_laterality = int((laterality_per_study.apply(len) == 0).sum())
n_single_laterality = int((laterality_per_study.apply(len) == 1).sum())
n_conflicting_laterality = int((laterality_per_study.apply(len) > 1).sum())

print(f"Studies with no laterality info at all: {n_no_laterality}")
print(f"Studies with exactly one consistent laterality value: {n_single_laterality}")
print(f"Studies with CONFLICTING laterality across series: {n_conflicting_laterality}")

all_laterality_values = set()
for s in laterality_per_study:
    all_laterality_values |= s
print(f"\nDistinct laterality values observed: {all_laterality_values}")

unexpected_values = all_laterality_values - {"L", "R"}
if unexpected_values:
    print(f"WARNING: unexpected laterality values found: {unexpected_values}")

pct_coverage = 100 * (n_single_laterality + n_conflicting_laterality) / len(laterality_per_study) if len(laterality_per_study) else 0
print(f"\nLaterality coverage (any value present): {pct_coverage:.2f}%")

unexpected_bodyparts = set()
for s in bodypart_per_study:
    unexpected_bodyparts |= s
unexpected_bodyparts -= {"KNEE", "EXTREMITY"}
if unexpected_bodyparts:
    print(f"\nWARNING: unexpected BodyPartExamined values found: {unexpected_bodyparts}")

### Bilateral / Ambiguous Study Investigation

Studies flagged below by the metadata scan (conflicting laterality) are candidates for visual verification, not confirmed bilateral cases yet.

In [ ]:
conflicting_ids = []
if n_conflicting_laterality > 0:
    conflicting_ids = laterality_per_study[laterality_per_study.apply(len) > 1].index.tolist()
    print(f"Studies flagged for potential bilateral examination (conflicting laterality across series): {len(conflicting_ids)}")
    print(conflicting_ids[:20], "..." if len(conflicting_ids) > 20 else "")
else:
    print("No studies flagged by laterality conflict.")

print(f"\nStudies with no laterality info (ambiguous, not necessarily bilateral): {n_no_laterality}")

In [ ]:
# Corrected slice-ordering helper: does NOT trust filename order.
# Falls back to ImagePositionPatient projection if InstanceNumber is missing/non-unique,
# and explicitly warns rather than silently guessing if neither is usable.
def get_ordered_series_files(series_path):
    files = glob.glob(os.path.join(series_path, "*.dcm"))
    infos = []
    for f in files:
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        infos.append((f, getattr(ds, "InstanceNumber", None), getattr(ds, "ImagePositionPatient", None)))

    instance_numbers = [i[1] for i in infos]
    if all(n is not None for n in instance_numbers) and len(set(instance_numbers)) == len(instance_numbers):
        infos.sort(key=lambda x: x[1])
        return [i[0] for i in infos], "InstanceNumber"

    positions = [i[2] for i in infos]
    if all(p is not None for p in positions):
        positions_arr = np.array(positions, dtype=float)
        axis = int(positions_arr.var(axis=0).argmax())
        infos.sort(key=lambda x: x[2][axis])
        return [i[0] for i in infos], f"ImagePositionPatient[axis={axis}]"

    print(f"WARNING: cannot reliably order {series_path} -- neither InstanceNumber nor "
          "ImagePositionPatient is fully populated. Falling back to filename order; treat any "
          "'middle slice' from this series as arbitrary, not anatomically central.")
    return sorted(files), "filename (UNRELIABLE)"

# Visual verification for up to 5 flagged studies (metadata-flagged, not cherry-picked)
studies_to_check = conflicting_ids[:5]
for study in studies_to_check:
    study_path = os.path.join(DICOM_ROOT, study)
    series_dirs = [d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d))]
    fig, axes = plt.subplots(1, len(series_dirs), figsize=(5 * len(series_dirs), 5))
    if len(series_dirs) == 1:
        axes = [axes]
    for ax, series in zip(axes, series_dirs):
        ordered_files, method = get_ordered_series_files(os.path.join(study_path, series))
        if not ordered_files:
            continue
        mid_path = ordered_files[len(ordered_files) // 2]
        ds = pydicom.dcmread(mid_path)
        ax.imshow(ds.pixel_array, cmap="gray")
        ax.set_title(f"{series[:12]}...\n(ordered by {method})")
        ax.axis("off")
    fig.suptitle(f"Flagged study {study[:20]}... -- potential bilateral, visual check")
    plt.tight_layout()
    plt.show()

if not studies_to_check:
    print("No flagged studies to visually verify.")

**Note on Section 12's original visualization (`show_study_planes`):** that function selects a "representative" slice using `dcm_paths[len(dcm_paths) // 2]` after a plain filename-sorted glob. Given Section 8/13 already established file-listing order ≠ anatomical order, that slice should be understood as an **arbitrary** representative slice, not a verified anatomical middle — the visualizations in Section 12 were useful for a first look at image content, but not for anatomical-position claims. `get_ordered_series_files` above is the corrected approach for any visualization where slice position actually matters (as it does here, for bilateral verification).

### Full-Dataset Manufacturer Analysis

In [ ]:
print("Manufacturer distribution (full dataset, from representative-file scan):")
print(study_level_df["Manufacturer"].value_counts(dropna=False))

print("\nManufacturerModelName distribution:")
print(study_level_df["ManufacturerModelName"].value_counts(dropna=False))

### Validation Implications

In [ ]:
print("=== VALIDATION IMPLICATION (Patient) ===")
if n_with_patient == n_total and n_multi == 0 and len(inconsistent_patient_studies) == 0:
    print(
        "Case A: every patient has exactly one study. A non-grouped split carries no leakage risk "
        "from repeated patients specifically. Other leakage vectors (e.g. near-duplicate imaging) "
        "are not addressed by this finding alone."
    )
elif n_with_patient == n_total and n_multi > 0:
    print(
        f"Case B: {n_multi} patients ({pct_multi_studies:.2f}% of studies) have more than one study. "
        "Patient-grouped validation (e.g. GroupKFold on PatientID) is REQUIRED -- a random or "
        "stratified-only split risks leaking a patient's other study into a different fold."
    )
else:
    print(
        f"Case C: PatientID coverage is incomplete or inconsistent "
        f"({n_with_patient}/{n_total} populated, {len(inconsistent_patient_studies)} inconsistent studies). "
        "A fully patient-grouped split may not be reliably implementable for every study. Consider "
        "grouping where PatientID is available and clean, and explicitly documenting accepted leakage "
        "risk for the remainder, rather than silently falling back to an ungrouped split."
    )

print("\n=== LATERALITY / TARGET-SEMANTICS IMPLICATION ===")
if n_conflicting_laterality == 0 and pct_coverage > 95:
    print(
        "Strong evidence across the full dataset: laterality is consistent within every study where "
        "populated, with high coverage. Target labels can be treated as single-knee-per-study with "
        "high confidence, though the small uncovered fraction remains formally unconfirmed."
    )
elif n_conflicting_laterality > 0:
    print(
        f"{n_conflicting_laterality} studies ({100*n_conflicting_laterality/n_total:.2f}% of the dataset) "
        "show conflicting laterality across their series -- flagged as potential bilateral studies. "
        "Target semantics CANNOT be safely assumed single-knee-per-study across the whole dataset "
        "without per-study handling for these flagged cases (see visual verification above)."
    )
else:
    print(
        "Laterality coverage is incomplete across the dataset; evidence leans toward single-knee-per-"
        "study for the covered fraction but is not conclusive dataset-wide."
    )

### Full-Dataset Data Quality Summary

In [ ]:
full_dataset_quality = {
    "studies_with_no_readable_header": len(failed_reads),
    "studies_missing_patientid": n_total - n_with_patient,
    "studies_with_inconsistent_patientid": len(inconsistent_patient_studies),
    "studies_with_no_laterality": n_no_laterality,
    "studies_with_conflicting_laterality": n_conflicting_laterality,
    "unexpected_laterality_values": list(unexpected_values) if unexpected_values else None,
    "unexpected_bodypart_values": list(unexpected_bodyparts) if unexpected_bodyparts else None,
}

print("Full-dataset DICOM header quality summary:")
for k, v in full_dataset_quality.items():
    flag = "  <-- non-zero/non-empty" if v not in (0, None, []) else ""
    print(f"  {k}: {v}{flag}")

In [ ]:
def normalize_laterality(series):
    mapped = series.dropna().apply(lambda v: LATERALITY_NORMALIZE.get(v, v))
    return set(v for v in mapped if v is not None and v != "")

laterality_per_study_normalized = series_level_df.groupby("StudyInstanceUID")["Laterality"].apply(normalize_laterality)

n_no_laterality_norm = int(laterality_per_study_normalized.apply(len).eq(0).sum())
n_single_laterality_norm = int(laterality_per_study_normalized.apply(len).eq(1).sum())
n_conflicting_laterality_norm = int(laterality_per_study_normalized.apply(len).gt(1).sum())
n_explicit_bilateral = int(laterality_per_study_normalized.apply(lambda s: s == {"B"}).sum())

print(f"After normalization -- no laterality info: {n_no_laterality_norm}")
print(f"After normalization -- single consistent value: {n_single_laterality_norm}")
print(f"After normalization -- genuinely conflicting: {n_conflicting_laterality_norm}")
print(f"Studies explicitly tagged 'B' only: {n_explicit_bilateral}")

all_values_norm = set()
for s in laterality_per_study_normalized:
    all_values_norm |= s
print(f"\nDistinct normalized values: {all_values_norm}")
still_unexpected = all_values_norm - {"L", "R", "B"}
if still_unexpected:
    print(f"WARNING: still-unrecognized values: {still_unexpected}")

In [ ]:
missing_laterality_studies = laterality_per_study_normalized[laterality_per_study_normalized.apply(len) == 0].index
missing_mask = study_level_df["StudyInstanceUID"].isin(missing_laterality_studies)
print("Manufacturer distribution among studies with NO laterality tag:")
print(study_level_df[missing_mask]["Manufacturer"].value_counts(dropna=False))
print("\nManufacturer distribution among studies WITH laterality tag (for comparison):")
print(study_level_df[~missing_mask]["Manufacturer"].value_counts(dropna=False))

### Visual Verification of the 26 Laterality-Exception Studies

Uses the corrected (post-normalization) conflict list, not the earlier buggy 83-study list from Cell 43. Includes a geometric pre-classification step (comparing `ImagePositionPatient` between the conflicting-laterality series groups) before visual inspection — large physical separation between the two laterality-tagged groups suggests a genuine bilateral exam; near-identical position suggests the same knee tagged inconsistently. This narrows down which of the 26 cases most need actual eyeballing.

In [ ]:
conflicting_ids_norm = laterality_per_study_normalized[laterality_per_study_normalized.apply(len) > 1].index.tolist()
explicit_bilateral_ids = laterality_per_study_normalized[
    laterality_per_study_normalized.apply(lambda s: s == {"B"})
].index.tolist()
flagged_studies = sorted(set(conflicting_ids_norm) | set(explicit_bilateral_ids))

print(f"Total flagged studies: {len(flagged_studies)}")
print(f"  L/R conflict: {len(conflicting_ids_norm)}")
print(f"  Explicit 'B' tag: {len(explicit_bilateral_ids)}")

In [ ]:
def get_series_position_centroid(series_path):
    files = glob.glob(os.path.join(series_path, "*.dcm"))
    positions = []
    for f in files:
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        pos = getattr(ds, "ImagePositionPatient", None)
        if pos is not None:
            positions.append([float(x) for x in pos])
    return np.mean(positions, axis=0) if positions else None

BILATERAL_DISTANCE_THRESHOLD_MM = 50.0  # plausible minimum real-world separation between two knees

results = []
for study in flagged_studies:
    study_path = os.path.join(DICOM_ROOT, study)
    series_dirs = [d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d))]
    is_explicit_b = study in explicit_bilateral_ids

    geometry_verdict = None
    if is_explicit_b and len(series_dirs) == 1:
        geometry_verdict = "N/A (single series, explicit B tag)"
    else:
        group_centroids = {}
        for series in series_dirs:
            f = sorted(glob.glob(os.path.join(study_path, series, "*.dcm")))
            if not f:
                continue
            ds = pydicom.dcmread(f[0], stop_before_pixels=True)
            lat_raw = getattr(ds, "Laterality", None)
            lat = LATERALITY_NORMALIZE.get(lat_raw, lat_raw) if lat_raw else None
            if lat is None:
                continue
            centroid = get_series_position_centroid(os.path.join(study_path, series))
            if centroid is not None:
                group_centroids.setdefault(lat, []).append(centroid)

        if len(group_centroids) >= 2 and all(len(v) > 0 for v in group_centroids.values()):
            group_means = {k: np.mean(v, axis=0) for k, v in group_centroids.items()}
            keys = list(group_means.keys())
            dist = np.linalg.norm(group_means[keys[0]] - group_means[keys[1]])
            geometry_verdict = (
                f"Geometry suggests TRUE BILATERAL (separation={dist:.1f}mm)"
                if dist >= BILATERAL_DISTANCE_THRESHOLD_MM
                else f"Geometry suggests SAME-LOCATION / likely tagging artifact (separation={dist:.1f}mm)"
            )
        else:
            geometry_verdict = "Insufficient positional metadata for geometric pre-classification"

    results.append({
        "StudyInstanceUID": study,
        "flag_type": "explicit_B" if is_explicit_b else "L/R_conflict",
        "n_series": len(series_dirs),
        "geometry_verdict": geometry_verdict,
        "final_classification": "PENDING VISUAL REVIEW",
    })

verification_df = pd.DataFrame(results)
print(verification_df.to_string(index=False))
print(f"\nGeometry-suggested TRUE BILATERAL: {(verification_df['geometry_verdict'].str.contains('TRUE BILATERAL')).sum()}")
print(f"Geometry-suggested artifact: {(verification_df['geometry_verdict'].str.contains('artifact')).sum()}")
print(f"Insufficient/N-A: {(~verification_df['geometry_verdict'].str.contains('TRUE BILATERAL|artifact')).sum()}")

In [ ]:
# Visual pass over ALL 26 flagged studies (not a sample — this is the full exception list).
for study in flagged_studies:
    study_path = os.path.join(DICOM_ROOT, study)
    series_dirs = [d for d in os.listdir(study_path) if os.path.isdir(os.path.join(study_path, d))]
    fig, axes = plt.subplots(1, len(series_dirs), figsize=(5 * len(series_dirs), 5))
    if len(series_dirs) == 1:
        axes = [axes]
    for ax, series in zip(axes, series_dirs):
        ordered_files, method = get_ordered_series_files(os.path.join(study_path, series))
        if not ordered_files:
            continue
        mid_path = ordered_files[len(ordered_files) // 2]
        ds = pydicom.dcmread(mid_path)
        lat = getattr(ds, "Laterality", "?")
        ax.imshow(ds.pixel_array, cmap="gray")
        ax.set_title(f"{series[:12]}...\nLaterality={lat}, ordered by {method}")
        ax.axis("off")
    geom = verification_df.loc[verification_df["StudyInstanceUID"] == study, "geometry_verdict"].values[0]
    fig.suptitle(f"{study[:24]}... | {geom}")
    plt.tight_layout()
    plt.show()

**After reviewing the images above, manually update `verification_df["final_classification"]` for each study** with one of: `clearly unilateral L`, `clearly unilateral R`, `clearly bilateral`, `metadata artifact`, `ambiguous`. Example:

In [ ]:
manual_classifications = {
    "1.2.826.0.1.3680043.8.498.11596665780617412984766822528862741584": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.12130868132537772155475981935868588761": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.13030662056306690969977140713179461363": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.38056024157303576597111203925091929489": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.38303182087206935163155565441234078389": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.42396216793489016293032290339618185275": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.78119223228067919465880137763593966456": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.83030371526667747898709613600825477966": "clearly bilateral",
    "1.2.826.0.1.3680043.8.498.11924999889245769866426422681253524920": "likely bilateral (moderate-high confidence)",
    "1.2.826.0.1.3680043.8.498.13242536312535334908597493894251783308": "likely bilateral (moderate-high confidence)",
    "1.2.826.0.1.3680043.8.498.12333996498811456012744466468462396081": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.22407175411826914385667795929161924645": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.40898220266193841898869905714214454387": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.52204390785696944340286127542242341202": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.57239450717434070615919278253974598533": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.58811528407321581206563412079643094538": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.66210455934011506415244876770725282769": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.70124572708965486791612654433965515063": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.73215265387578908341609746559916587894": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.88274753211076645976496220746740420894": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.88752039058074948027322975455607258941": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.91125742927944703615970502025043944089": "likely bilateral (lower confidence)",
    "1.2.826.0.1.3680043.8.498.30248682294786626375369374249874501600": "metadata artifact (lower confidence)",
    "1.2.826.0.1.3680043.8.498.46172451572836863687073454997158931662": "ambiguous",
    "1.2.826.0.1.3680043.8.498.62420632149863295140175740643640707423": "metadata artifact / inconsistency",
    "1.2.826.0.1.3680043.8.498.11504079342934513987748646905909982588": "metadata artifact / ambiguous",
}
verification_df["final_classification"] = verification_df["StudyInstanceUID"].map(manual_classifications)
print(verification_df["final_classification"].value_counts())